<img src="https://courseware.decoded.com/images/decoded/logo-decoded.png" align="left" width="100px"/>

# Regression model selection and validation

**Learner notebook — standalone modelling workflow**

In this notebook, you will train and compare several regression algorithms, use cross-validation, tune hyperparameters and evaluate a final model on unseen data.

> **Starting point:** The dataset has already been cleaned and prepared in the previous module. This notebook focuses only on modelling and evaluation.

---

<small>© COPYRIGHT 2026 DECODED LIMITED. ALL RIGHTS RESERVED. Web: https://decoded.com/</small>

## Why this matters

A regression model predicts a numerical value. In this scenario, the target is bike-rental demand.

A reliable model could help an operations team plan bike availability, schedule redistribution and prepare for periods of high or low demand.

The aim is not to find a model that performs well only on the data it has already seen. We want a model that can make useful predictions for new observations.

## Learning outcomes

By the end of this notebook, you will be able to:

1. train several regression algorithms
2. generate predictions for unseen data
3. compare MAE, RMSE and $R^2$
4. use cross-validation to assess model stability
5. tune Random Forest hyperparameters
6. evaluate a final model on a protected test set
7. explain model performance in a workplace context

## Learning journey

1. Load the prepared dataset  
2. Define the features and target  
3. Create training and test sets  
4. Establish a baseline  
5. Train different regression models  
6. Compare evaluation metrics  
7. Cross-validate the models  
8. Tune a Random Forest  
9. Evaluate the final model  
10. Diagnose errors and reflect

## 1. Import the libraries

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_validate, train_test_split
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

print("Libraries imported.")

## 2. Load the prepared dataset

The code looks for a prepared bike-rental CSV. If the file is not available, it creates a synthetic modelling-ready dataset so the notebook can still be completed.

The fallback data is for learning only.

In [ ]:
candidate_paths = [
    Path("seoulbikerentals_prepared.csv"),
    Path("/mnt/data/seoulbikerentals_prepared.csv"),
    Path("seoulbikerentals.csv"),
    Path("/mnt/data/seoulbikerentals.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is not None:
    data = pd.read_csv(data_path)
    print(f"Loaded: {data_path}")
else:
    rng = np.random.default_rng(RANDOM_STATE)
    n_rows = 1200

    hour = rng.integers(0, 24, n_rows)
    temperature = rng.normal(13, 9, n_rows)
    humidity = np.clip(rng.normal(58, 18, n_rows), 10, 100)
    wind_speed = np.clip(rng.normal(2.2, 1.0, n_rows), 0, None)
    rainfall = np.where(rng.random(n_rows) < 0.12, rng.gamma(1.5, 3.0, n_rows), 0)
    is_holiday = rng.choice([0, 1], n_rows, p=[0.93, 0.07])
    is_functioning_day = rng.choice([1, 0], n_rows, p=[0.97, 0.03])

    commute_peak = np.where(
        ((hour >= 7) & (hour <= 9)) | ((hour >= 16) & (hour <= 19)),
        280,
        0,
    )

    demand = (
        320
        + 18 * temperature
        - 3 * humidity
        - 32 * rainfall
        + commute_peak
        - 100 * is_holiday
        - 500 * (1 - is_functioning_day)
        + rng.normal(0, 95, n_rows)
    )

    data = pd.DataFrame({
        "hour": hour,
        "temperature_c": temperature.round(1),
        "humidity_pct": humidity.round(1),
        "wind_speed_m_s": wind_speed.round(1),
        "rainfall_mm": rainfall.round(1),
        "is_holiday": is_holiday,
        "is_functioning_day": is_functioning_day,
        "rented_bike_count": np.clip(demand, 0, None).round().astype(int),
    })
    print("Prepared CSV not found. Using synthetic practice data.")

print("Dataset shape:", data.shape)
data.head()

### Confirm the dataset is modelling-ready

The previous module should already have handled missing values, text categories and formatting issues.

This check does not preprocess the data. It confirms whether the notebook can proceed.

In [ ]:
print("Missing values:", int(data.isna().sum().sum()))
print("Non-numeric columns:", data.select_dtypes(exclude="number").columns.tolist())

In [ ]:
if data.isna().sum().sum() > 0:
    raise ValueError(
        "The dataset still contains missing values. Complete preprocessing in the previous module first."
    )

non_numeric_columns = data.select_dtypes(exclude="number").columns.tolist()
if non_numeric_columns:
    raise ValueError(
        "The dataset still contains non-numeric columns: "
        f"{non_numeric_columns}. Encode them in the previous module first."
    )

print("The dataset is ready for modelling.")

## 3. Define the features and target

In [ ]:
target_column = "rented_bike_count"

if target_column not in data.columns:
    raise ValueError(
        f"Expected target column '{target_column}' was not found."
    )

In [ ]:
X = data.drop(columns=[target_column])
y = data[target_column]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
X.head()

In [ ]:
y.describe().to_frame("rented_bike_count")

### Learner checkpoint

Before training a model, answer these questions:

1. What value is the model trying to predict?
2. Which columns will the model use as inputs?
3. Why must the target column be removed from `X`?

## 4. Create training and test sets

The training set is used to fit and compare models.

The test set is kept separate until the end. It provides a final check of performance on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

In [ ]:
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 5. Establish a baseline

A baseline gives us a minimum standard to beat.

`DummyRegressor` predicts the mean training target for every row. It does not learn relationships between the features and bike demand.

In [ ]:
baseline_model = DummyRegressor(strategy="mean")

In [ ]:
baseline_model.fit(X_train, y_train)

In [ ]:
baseline_predictions = baseline_model.predict(X_test)

In [ ]:
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = mean_squared_error(y_test, baseline_predictions) ** 0.5
baseline_r2 = r2_score(y_test, baseline_predictions)

print(f"Baseline MAE:  {baseline_mae:.2f}")
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Baseline R²:   {baseline_r2:.3f}")

## 6. Understand the regression metrics

| Metric | What it tells us | Why it is useful |
|---|---|---|
| **MAE** | The average absolute difference between actual and predicted values | Easy to explain in bike counts |
| **RMSE** | The typical error, with extra weight given to large mistakes | Useful when large forecasting errors are costly |
| **$R^2$** | How much better the model explains variation than predicting the mean | Useful for comparing overall fit |

Use the metrics together. A single metric does not tell the whole story.

Avoid relying on MAPE here because actual bike demand may be zero. Percentage errors become unstable when the actual value is zero or close to zero.

## 7. Train a Linear Regression model

Linear Regression estimates a straight-line relationship between the features and the target.

It is useful as an interpretable benchmark, even when we expect some relationships to be non-linear.

In [ ]:
linear_model = LinearRegression()

In [ ]:
linear_model.fit(X_train, y_train)

In [ ]:
linear_predictions = linear_model.predict(X_test)

In [ ]:
linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_rmse = mean_squared_error(y_test, linear_predictions) ** 0.5
linear_r2 = r2_score(y_test, linear_predictions)

print(f"Linear Regression MAE:  {linear_mae:.2f}")
print(f"Linear Regression RMSE: {linear_rmse:.2f}")
print(f"Linear Regression R²:   {linear_r2:.3f}")

## 8. Train a Ridge Regression model

Ridge Regression is a regularised version of Linear Regression.

Regularisation discourages very large coefficients. This can improve stability when features overlap in the information they provide.

In [ ]:
ridge_model = Ridge(alpha=1.0)

In [ ]:
ridge_model.fit(X_train, y_train)

In [ ]:
ridge_predictions = ridge_model.predict(X_test)

In [ ]:
ridge_mae = mean_absolute_error(y_test, ridge_predictions)
ridge_rmse = mean_squared_error(y_test, ridge_predictions) ** 0.5
ridge_r2 = r2_score(y_test, ridge_predictions)

print(f"Ridge Regression MAE:  {ridge_mae:.2f}")
print(f"Ridge Regression RMSE: {ridge_rmse:.2f}")
print(f"Ridge Regression R²:   {ridge_r2:.3f}")

## 9. Train a Decision Tree model

A Decision Tree learns a sequence of rules.

It can capture non-linear relationships, but a deep tree can fit the training data too closely.

In [ ]:
tree_model = DecisionTreeRegressor(random_state=RANDOM_STATE)

In [ ]:
tree_model.fit(X_train, y_train)

In [ ]:
tree_predictions = tree_model.predict(X_test)

In [ ]:
tree_mae = mean_absolute_error(y_test, tree_predictions)
tree_rmse = mean_squared_error(y_test, tree_predictions) ** 0.5
tree_r2 = r2_score(y_test, tree_predictions)

print(f"Decision Tree MAE:  {tree_mae:.2f}")
print(f"Decision Tree RMSE: {tree_rmse:.2f}")
print(f"Decision Tree R²:   {tree_r2:.3f}")

## 10. Train a Random Forest model

A Random Forest trains many decision trees and averages their predictions.

Averaging usually makes the model more stable than a single tree.

In [ ]:
forest_model = RandomForestRegressor(
    n_estimators=150,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

In [ ]:
forest_model.fit(X_train, y_train)

In [ ]:
forest_predictions = forest_model.predict(X_test)

In [ ]:
forest_mae = mean_absolute_error(y_test, forest_predictions)
forest_rmse = mean_squared_error(y_test, forest_predictions) ** 0.5
forest_r2 = r2_score(y_test, forest_predictions)

print(f"Random Forest MAE:  {forest_mae:.2f}")
print(f"Random Forest RMSE: {forest_rmse:.2f}")
print(f"Random Forest R²:   {forest_r2:.3f}")

## 11. Train a Gradient Boosting model

Gradient Boosting builds trees one after another.

Each new tree focuses on correcting errors made by the previous trees.

In [ ]:
boosting_model = GradientBoostingRegressor(random_state=RANDOM_STATE)

In [ ]:
boosting_model.fit(X_train, y_train)

In [ ]:
boosting_predictions = boosting_model.predict(X_test)

In [ ]:
boosting_mae = mean_absolute_error(y_test, boosting_predictions)
boosting_rmse = mean_squared_error(y_test, boosting_predictions) ** 0.5
boosting_r2 = r2_score(y_test, boosting_predictions)

print(f"Gradient Boosting MAE:  {boosting_mae:.2f}")
print(f"Gradient Boosting RMSE: {boosting_rmse:.2f}")
print(f"Gradient Boosting R²:   {boosting_r2:.3f}")

## 12. Compare the first test results

In [ ]:
initial_results = pd.DataFrame([
    ["Mean baseline", baseline_mae, baseline_rmse, baseline_r2],
    ["Linear Regression", linear_mae, linear_rmse, linear_r2],
    ["Ridge Regression", ridge_mae, ridge_rmse, ridge_r2],
    ["Decision Tree", tree_mae, tree_rmse, tree_r2],
    ["Random Forest", forest_mae, forest_rmse, forest_r2],
    ["Gradient Boosting", boosting_mae, boosting_rmse, boosting_r2],
], columns=["model", "MAE", "RMSE", "R2"])

initial_results.sort_values("RMSE").round(3)

### Important

This first comparison helps us understand the models, but we should not select a final model from one split alone.

The next section uses cross-validation on the training data to produce a more reliable comparison.

## 13. Set up cross-validation

Five-fold cross-validation repeats the training and validation process five times.

Each fold uses a different section of the training data for validation. This helps us assess both average performance and consistency.

In [ ]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

In [ ]:
scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

## 14. Cross-validate one model

Start with Linear Regression so that the structure of `cross_validate()` is clear.

In [ ]:
linear_cv_scores = cross_validate(
    linear_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
)

In [ ]:
linear_cv_scores

Scikit-learn returns MAE and RMSE as negative values because its scoring system assumes that larger values are better.

We reverse the sign before reporting the errors.

In [ ]:
print(f"Mean CV MAE:  {-linear_cv_scores['test_mae'].mean():.2f}")
print(f"Mean CV RMSE: {-linear_cv_scores['test_rmse'].mean():.2f}")
print(f"Mean CV R²:   {linear_cv_scores['test_r2'].mean():.3f}")

## 15. Cross-validate all candidate models

In [ ]:
models = {
    "Linear Regression": linear_model,
    "Ridge Regression": ridge_model,
    "Decision Tree": tree_model,
    "Random Forest": forest_model,
    "Gradient Boosting": boosting_model,
}

In [ ]:
cv_rows = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )

    cv_rows.append({
        "model": model_name,
        "mean_MAE": -scores["test_mae"].mean(),
        "std_MAE": scores["test_mae"].std(),
        "mean_RMSE": -scores["test_rmse"].mean(),
        "std_RMSE": scores["test_rmse"].std(),
        "mean_R2": scores["test_r2"].mean(),
        "std_R2": scores["test_r2"].std(),
    })

In [ ]:
cv_results = pd.DataFrame(cv_rows).sort_values("mean_RMSE")
cv_results.round(3)

In [ ]:
plot_data = cv_results.sort_values("mean_RMSE", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(plot_data["model"], plot_data["mean_RMSE"], xerr=plot_data["std_RMSE"])
plt.xlabel("Cross-validated RMSE")
plt.ylabel("Model")
plt.title("Lower RMSE is better")
plt.show()

### Learner checkpoint

1. Which model has the lowest mean RMSE?
2. Which model is most consistent across folds?
3. Are the differences large enough to matter?
4. Would you choose the most accurate model if it were much harder to explain?

## 16. Hyperparameter tuning

A parameter is learned from the data. A hyperparameter is selected before training.

For a Random Forest:

- `n_estimators` controls the number of trees
- `max_depth` limits tree depth
- `min_samples_split` controls the minimum data required to split a node
- `max_features` controls how many features are considered at each split

In [ ]:
parameter_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 8, 16],
    "min_samples_split": [2, 5],
    "max_features": ["sqrt", 1.0],
}

In [ ]:
tuning_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

In [ ]:
grid_search = GridSearchCV(
    estimator=tuning_model,
    param_grid=parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
print("Best parameters:")
print(grid_search.best_params_)
print(f"Best cross-validated RMSE: {-grid_search.best_score_:.2f}")

## 17. Review the tuning results

In [ ]:
tuning_results = pd.DataFrame(grid_search.cv_results_)

In [ ]:
tuning_summary = (
    tuning_results[[
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_n_estimators",
        "param_max_depth",
        "param_min_samples_split",
        "param_max_features",
    ]]
    .assign(mean_CV_RMSE=lambda frame: -frame["mean_test_score"])
    .sort_values("rank_test_score")
    .drop(columns="mean_test_score")
    .head(10)
)

tuning_summary.round(3)

## 18. Evaluate the final model on the test set

The test set has not been used during cross-validation or tuning.

We now use it once to estimate how the selected model performs on unseen data.

In [ ]:
final_model = grid_search.best_estimator_

In [ ]:
final_predictions = final_model.predict(X_test)

In [ ]:
final_mae = mean_absolute_error(y_test, final_predictions)
final_rmse = mean_squared_error(y_test, final_predictions) ** 0.5
final_r2 = r2_score(y_test, final_predictions)

print(f"Final MAE:  {final_mae:.2f}")
print(f"Final RMSE: {final_rmse:.2f}")
print(f"Final R²:   {final_r2:.3f}")

In [ ]:
final_comparison = pd.DataFrame([
    ["Mean baseline", baseline_mae, baseline_rmse, baseline_r2],
    ["Tuned Random Forest", final_mae, final_rmse, final_r2],
], columns=["model", "MAE", "RMSE", "R2"])

final_comparison.round(3)

### Interpret the final metrics

A strong interpretation should mention all three measures:

- MAE describes the typical error in bike counts
- RMSE highlights the effect of larger mistakes
- $R^2$ shows improvement over predicting the mean

Do not describe $R^2$ as a percentage accuracy score.

## 19. Plot actual and predicted values

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, final_predictions, alpha=0.6)

minimum = min(y_test.min(), final_predictions.min())
maximum = max(y_test.max(), final_predictions.max())
plt.plot([minimum, maximum], [minimum, maximum], linestyle="--")

plt.xlabel("Actual bike demand")
plt.ylabel("Predicted bike demand")
plt.title("Actual versus predicted demand")
plt.show()

## 20. Examine residuals

A residual is:

```text
actual value - predicted value
```

- positive residual: the model under-predicted
- negative residual: the model over-predicted

In [ ]:
residuals = y_test - final_predictions

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(final_predictions, residuals, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted bike demand")
plt.ylabel("Residual")
plt.title("Residuals versus predictions")
plt.show()

## 21. Inspect the largest errors

In [ ]:
error_review = X_test.copy()
error_review["actual_demand"] = y_test
error_review["predicted_demand"] = final_predictions
error_review["residual"] = residuals
error_review["absolute_error"] = np.abs(residuals)

In [ ]:
error_review.sort_values("absolute_error", ascending=False).head(10)

### Learner task

Review the ten largest errors.

1. Do the cases share any visible conditions?
2. Does the model mainly under-predict or over-predict them?
3. What additional information might improve these predictions?
4. What could be the operational impact of these mistakes?

## 22. Check for overfitting

In [ ]:
train_predictions = final_model.predict(X_train)

In [ ]:
generalisation_check = pd.DataFrame([
    [
        "Training",
        mean_absolute_error(y_train, train_predictions),
        mean_squared_error(y_train, train_predictions) ** 0.5,
        r2_score(y_train, train_predictions),
    ],
    [
        "Test",
        final_mae,
        final_rmse,
        final_r2,
    ],
], columns=["dataset", "MAE", "RMSE", "R2"])

generalisation_check.round(3)

A large gap between training and test performance may suggest overfitting.

A small gap does not prove that the model will perform well in every future setting, but it is a useful diagnostic.

## 23. Responsible interpretation

Before using the model operationally, consider:

- whether the training data represents future operating conditions
- whether any feature would be unavailable at prediction time
- whether travel patterns could change over time
- the impact of under-predicting demand
- who will review unusual forecasts
- how performance will be monitored

Model selection is a technical and business decision.

## Knowledge check

1. Why do we keep a separate test set?
2. Why is a baseline useful?
3. What is the difference between MAE and RMSE?
4. Why can $R^2$ be misleading on its own?
5. What does cross-validation add to a single train/test split?
6. What is the difference between a parameter and a hyperparameter?
7. Why must tuning use only the training data?
8. What evidence would you need before deploying this model?

## Key takeaways

- Begin with a baseline.
- Train different algorithms because they make different assumptions.
- Use MAE, RMSE and $R^2$ together.
- Use cross-validation to compare average performance and stability.
- Tune hyperparameters on the training data only.
- Use the test set once for final evaluation.
- Investigate individual errors as well as summary scores.
- Select a model using performance, stability, explainability and operational impact.

## What to learn next

Next, explore:

- feature importance
- model explainability
- time-aware validation
- prediction intervals
- model monitoring
- advanced workflows that combine preprocessing and modelling